# 第 1 周练习 —— 技术问题解释器（流式）

## 练习目标（理念）

提出任意技术问题，并从两类模型获得**流式（streaming）**解释：

- **前沿云端**：`gpt-4o-mini`
- **本地**：`llama3.2`（经 Ollama）

同一函数驱动两端，因为 Ollama 暴露了 **OpenAI 兼容 API**（`/v1`）。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions + `stream=True` | `explain(...)` 里逐块 `update_display` |
| `messages`（system / user） | 常量 `SYSTEM` + 用户 `question` |
| 双客户端 | `frontier`（云端）与 `local`（Ollama） |

## 怎么跑

1. 准备 `.env` 中的 `OPENAI_API_KEY`；本地需 Ollama 已拉取 `llama3.2`
2. 依次运行导入/函数/提问单元格，再分别跑「前沿」与「本地」两格对比


In [1]:
# ========== 导入、环境、双客户端与 system 指令 ==========

# 从 openai 导入 OpenAI 客户端类：同一套 SDK 打云端或 Ollama 兼容端点
from openai import OpenAI
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量
from dotenv import load_dotenv
# Markdown / display / update_display：笔记本里流式刷新 Markdown 显示
from IPython.display import Markdown, display, update_display

# 加载 .env；override=True 表示用文件值覆盖进程里已有同名环境变量
load_dotenv(override=True)

# 云端模型名（字符串必须与 API 接受的 id 一致）
MODEL_GPT = "gpt-4o-mini"
# 本地 Ollama 模型名（需事先 ollama pull）
MODEL_LLAMA = "llama3.2"

# 默认客户端：读取环境变量 OPENAI_API_KEY
frontier = OpenAI()  # uses OPENAI_API_KEY
# 指向本机 Ollama 的 OpenAI 兼容端点；api_key 多为占位字符串
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # Ollama, OpenAI-compatible

# system 指令保留英文：影响模型回答风格，不翻译
SYSTEM = "You are a helpful technical tutor. Explain clearly and concisely, with a short example when useful. Use markdown."


In [2]:
# ========== 核心函数：流式提问并在笔记本里边生成边刷新 Markdown ==========

def explain(question, client, model):
    """Stream an explanation and render it live as markdown."""
    # 组装 Chat Completions 的 messages：system 定风格，user 放问题
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": question},
    ]
    # stream=True：服务端持续返回增量 delta，而不是等整段生成完
    stream = client.chat.completions.create(model=model, messages=messages, stream=True)
    # reply：把每一块拼成完整文本，供 update_display 刷新
    reply = ""
    # 先占一个可更新的 Markdown 显示位（display_id=True）
    handle = display(Markdown(""), display_id=True)
    # 遍历流式事件：每来一块就追加并刷新同一显示位
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        update_display(Markdown(reply), display_id=handle.display_id)
    # 返回完整字符串，调用方可用 _ = ... 忽略
    return reply


In [3]:
# ========== 提问：改这里的字符串就能问新问题 ==========

# 发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""


### 前沿模型 — gpt-4o-mini

用云端 `frontier` 客户端 + `MODEL_GPT`，流式解释上面的 `question`。


In [4]:
# ========== 路径 A：云端 gpt-4o-mini 流式解释 ==========

# 把返回值赋给 _：表示「只要副作用（屏幕显示），不关心返回字符串」
_ = explain(question, frontier, MODEL_GPT)


The code you've provided is a Python expression that uses a combination of a generator, a set comprehension, and the `yield from` statement. Here’s a breakdown of what each part does:

### Breakdown of the Code

1. **Set Comprehension**:
   ```python
   {book.get("author") for book in books if book.get("author")}
   ```
   This part creates a set of authors from a collection of `books`. Here's how it works:
   - `for book in books`: Iterates over each `book` in the `books` list.
   - `book.get("author")`: Retrieves the value associated with the key "author" from the `book` dictionary.
   - `if book.get("author")`: This condition checks if the "author" exists and is truthy (i.e., not `None`, empty string, etc.). If it is, the author is included in the set.
   - The entire expression returns a set of unique authors because sets automatically handle duplicate values.

2. **Yield from**:
   ```python
   yield from ...
   ```
   The `yield from` statement is used to yield all values from a generator or iterable. In this context, it is applied to the set comprehension created earlier, effectively allowing you to yield each unique author one at a time.

### Purpose of the Code

This code snippet would typically be part of a generator function, and its purpose is to return each unique author from the list of books one at a time. Since it uses `yield from`, it allows for efficient handling of the authors without the need to store them in a separate list or array.

### Example

Here’s a simple example demonstrating how this code might be used:

```python
def get_unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Sample data
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": "Author A"},  # Duplicate author
    {"title": "Book 4"}  # No author
]

# Using the generator
for author in get_unique_authors(books):
    print(author)
```

### Output
```
Author A
Author B
```

In this example, the generator function `get_unique_authors` yields unique authors from the `books` list, demonstrating the use of `yield from` with a set comprehension effectively.

### 本地模型 — llama3.2（Ollama）

用 `local` 客户端（`localhost:11434/v1`）+ `MODEL_LLAMA`，同样走 `explain` 流式路径。


In [5]:
# ========== 路径 B：本地 llama3.2 流式解释 ==========

# 同一 explain 函数，只换 client 与 model，便于对比云端 vs 本地风格
_ = explain(question, local, MODEL_LLAMA)


**Generator Expression**
==========================

The given code uses a combination of two concepts: generator expression and `yield from`.

### Generator Expression

A generator expression is a compact way to create a generator, which is an iterable object that can be used to generate a sequence of values, one at a time.

Example:
```python
numbers = (x**2 for x in range(3))  # generator expression
print(next(numbers))  # prints 0
print(next(numbers))  # prints 1
print(next(numbers))  # prints 4
```

In this example, the generator expression creates a sequence of squares of numbers from `range(3)`.

### Yield from

`yield from` is a keyword used to pause the execution of a generator and delegate control to another generator. It allows you to yield values from another generator without having to implement a loop in Python code.

Example:
```python
def inner_generator():
    print("Inner gen")

    def small_gen():
        for i in [1, 2]:
            print(i)
    yield from small_gen()

inner_generator()
```

Output:
```
Inner gen
1
2
```

In this example, the `inner_generator` yields values from a nested generator called `small_gen`.

### Code Explanation

Now, let's apply these concepts to the original code:

```python
yield from {book.get("author") for book in books if book.get("author")}
```

This code uses a combination of `dict.get()` and a generator expression wrapped inside `{...}`. The dictionary values are obtained using `.get("author")`, and the result is an iterator.

The generator expression `(book.get("author")) for book in books if book.get("author")` yields values (i.e., authors) from each item `book` in the list `books`. However, due to a potential AttributeError when searching the dictionary value which may not exist, Python catches the error and uses `None` instead.

So, this code snippet essentially extracts all the author names from the `books` list while handling cases where authors might be missing.

Example Code with `List of Books**
```python
import json

class Book:
    def __init__(self, **kwargs):
        self.__dict__.update(kwargs)

books = [
    {"title": "Pride and Prejudice", "author":"Jane Austen"},
    {"title": "Wuthering Heights",},  # intentionally missing author field
    {"title": "Frankenstein", "author": "Mary Shelley"}
]

author_names = yield from {book.get("author") for book in books if book.get("author")}
print(author_names) # prints [None, None, 'Mary Shelley']
```